# Laboratorio 6 – Connect Four con Minimax
**CC3045 – Inteligencia Artificial**

- Task 2.1: lógica del juego + Minimax puro
- Task 2.2: Poda Alfa-Beta + comparación de nodos
- Task 2.3: función heurística + demos

---
## Task 2.1 – Lógica del juego y Minimax

In [9]:
import numpy as np
import random
import time

class Connect4:
    ROWS = 6
    COLS = 7
    EMPTY  = 0
    PLAYER = 1   # humano (MIN)
    AI     = 2   # agente (MAX)
    WIN_SCORE  =  1000
    LOSS_SCORE = -1000

    def __init__(self):
        self.board = np.zeros((self.ROWS, self.COLS), dtype=int)

    def copy(self):
        # se necesita copia del estado para cada rama recursiva
        new_game = Connect4()
        new_game.board = self.board.copy()
        return new_game

    def get_valid_columns(self):
        # actions(s): columna valida si la fila 0 esta libre
        return [c for c in range(self.COLS) if self.board[0][c] == self.EMPTY]

    def drop_piece(self, col, piece):
        # result(s, a): la ficha cae a la fila mas baja disponible
        for row in range(self.ROWS - 1, -1, -1):
            if self.board[row][col] == self.EMPTY:
                self.board[row][col] = piece
                return row
        return -1

    def check_win(self, piece):
        # is_terminal(s): revisa las 4 direcciones posibles
        b = self.board
        for r in range(self.ROWS):
            for c in range(self.COLS - 3):
                if all(b[r][c+i] == piece for i in range(4)):
                    return True
        for r in range(self.ROWS - 3):
            for c in range(self.COLS):
                if all(b[r+i][c] == piece for i in range(4)):
                    return True
        for r in range(3, self.ROWS):
            for c in range(self.COLS - 3):
                if all(b[r-i][c+i] == piece for i in range(4)):
                    return True
        for r in range(self.ROWS - 3):
            for c in range(self.COLS - 3):
                if all(b[r+i][c+i] == piece for i in range(4)):
                    return True
        return False

    def is_terminal(self):
        return (
            self.check_win(self.PLAYER) or
            self.check_win(self.AI) or
            len(self.get_valid_columns()) == 0
        )

    def print_board(self):
        symbols = {0: '.', 1: 'R', 2: 'Y'}
        print('  ' + '  '.join(str(c) for c in range(self.COLS)))
        for row in self.board:
            print('  ' + '  '.join(symbols[cell] for cell in row))
        print()

print('Clase Connect4 lista.')

Clase Connect4 lista.


In [10]:
# Heuristica evaluate_board() — se define antes de minimax() y minimax_ab()
# porque ambos la usan cuando depth == 0.
# Estrategia:
#   - +3 por ficha IA en columna central (col 3 tiene mas ventanas posibles)
#   - se escanean ventanas de 4 celdas en las 4 direcciones:
#       +1000  4 fichas IA
#        +50   3 fichas IA + 1 vacio
#         +5   2 fichas IA + 2 vacios
#        -80   3 fichas rival + 1 vacio  (bloquear)

def score_window(window, piece):
    opp = Connect4.PLAYER if piece == Connect4.AI else Connect4.AI
    score = 0
    cp = window.count(piece)
    ce = window.count(Connect4.EMPTY)
    co = window.count(opp)

    if   cp == 4:             score += 1000
    elif cp == 3 and ce == 1: score +=   50
    elif cp == 2 and ce == 2: score +=    5
    if   co == 3 and ce == 1: score -=   80  # -80 > +50 para priorizar defensa
    return score


def evaluate_board(board):
    # Eval(s): valor positivo = bueno para IA, negativo = bueno para humano
    ROWS  = Connect4.ROWS
    COLS  = Connect4.COLS
    piece = Connect4.AI
    score = 0

    center_col = [int(board[r][COLS // 2]) for r in range(ROWS)]
    score += center_col.count(piece) * 3

    for r in range(ROWS):
        for c in range(COLS - 3):
            score += score_window([board[r][c+i] for i in range(4)], piece)

    for c in range(COLS):
        for r in range(ROWS - 3):
            score += score_window([board[r+i][c] for i in range(4)], piece)

    for r in range(3, ROWS):
        for c in range(COLS - 3):
            score += score_window([board[r-i][c+i] for i in range(4)], piece)

    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            score += score_window([board[r+i][c+i] for i in range(4)], piece)

    return score

print('Heuristica lista.')

Heuristica lista.


In [11]:
# Minimax puro — complejidad O(b^d), sin poda
# Se limita a d=3 o d=4 porque sin poda se vuelve muy lento

nodes_minimax = 0

def minimax(game, depth, is_maximizing):
    # minimax(s) = utility(s)              si terminal
    # minimax(s) = max_a minimax(result)   si turno MAX
    # minimax(s) = min_a minimax(result)   si turno MIN
    global nodes_minimax
    nodes_minimax += 1

    if game.check_win(Connect4.AI):        return Connect4.WIN_SCORE
    if game.check_win(Connect4.PLAYER):    return Connect4.LOSS_SCORE
    if len(game.get_valid_columns()) == 0: return 0
    if depth == 0:                         return evaluate_board(game.board)

    valid_cols = game.get_valid_columns()

    if is_maximizing:
        best_value = float('-inf')
        for col in valid_cols:
            child = game.copy()
            child.drop_piece(col, Connect4.AI)
            value = minimax(child, depth - 1, False)
            best_value = max(best_value, value)
        return best_value
    else:
        best_value = float('inf')
        for col in valid_cols:
            child = game.copy()
            child.drop_piece(col, Connect4.PLAYER)
            value = minimax(child, depth - 1, True)
            best_value = min(best_value, value)
        return best_value


def get_best_move_minimax(game, depth=3):
    global nodes_minimax
    nodes_minimax = 0
    best_col   = None
    best_value = float('-inf')
    for col in game.get_valid_columns():
        child = game.copy()
        child.drop_piece(col, Connect4.AI)
        value = minimax(child, depth - 1, False)
        if value > best_value:
            best_value = value
            best_col   = col
    return best_col

print('Minimax puro listo (usar d=3 o d=4).')

Minimax puro listo (usar d=3 o d=4).


---
## Task 2.2 – Poda Alfa-Beta

Se agregan dos parámetros a Minimax:
- **α**: mejor valor que MAX encontró hasta ahora (inicia en −∞)
- **β**: mejor valor que MIN encontró hasta ahora (inicia en +∞)

Si en algún punto `α >= β`, la rama se corta porque el nodo padre nunca la elegiría. En el mejor caso la complejidad baja de O(b^d) a O(b^(d/2)), lo que permite explorar el doble de profundidad.

In [12]:
nodes_ab = 0

def minimax_ab(game, depth, alpha, beta, is_maximizing):
    # igual que minimax() pero se pasan alpha y beta para podar ramas
    global nodes_ab
    nodes_ab += 1

    if game.check_win(Connect4.AI):        return Connect4.WIN_SCORE
    if game.check_win(Connect4.PLAYER):    return Connect4.LOSS_SCORE
    if len(game.get_valid_columns()) == 0: return 0
    if depth == 0:                         return evaluate_board(game.board)

    valid_cols = game.get_valid_columns()

    if is_maximizing:
        best_value = float('-inf')
        for col in valid_cols:
            child = game.copy()
            child.drop_piece(col, Connect4.AI)
            value = minimax_ab(child, depth - 1, alpha, beta, False)
            best_value = max(best_value, value)
            alpha = max(alpha, best_value)
            if alpha >= beta:   # poda beta: MIN nunca elegira esta rama
                break
        return best_value
    else:
        best_value = float('inf')
        for col in valid_cols:
            child = game.copy()
            child.drop_piece(col, Connect4.PLAYER)
            value = minimax_ab(child, depth - 1, alpha, beta, True)
            best_value = min(best_value, value)
            beta = min(beta, best_value)
            if alpha >= beta:   # poda alfa: MAX nunca elegira esta rama
                break
        return best_value


def get_best_move_ab(game, depth=5):
    # en la raiz se llama con alpha=-inf, beta=+inf
    global nodes_ab
    nodes_ab = 0
    best_col   = None
    best_value = float('-inf')
    alpha      = float('-inf')
    beta       = float('inf')

    for col in game.get_valid_columns():
        child = game.copy()
        child.drop_piece(col, Connect4.AI)
        value = minimax_ab(child, depth - 1, alpha, beta, False)
        if value > best_value:
            best_value = value
            best_col   = col
        alpha = max(alpha, best_value)
    return best_col

print('Alfa-Beta listo (usar d=5 o d=6).')

Alfa-Beta listo (usar d=5 o d=6).


In [13]:
# Comparacion Minimax vs Alfa-Beta sobre el mismo tablero

demo_game = Connect4()
moves = [(3, Connect4.AI), (3, Connect4.PLAYER), (2, Connect4.AI),
         (4, Connect4.PLAYER), (2, Connect4.AI), (5, Connect4.PLAYER),
         (1, Connect4.AI), (0, Connect4.PLAYER)]
for col, piece in moves:
    demo_game.drop_piece(col, piece)

print('Tablero usado para la comparacion:')
demo_game.print_board()

DEPTH_COMPARE = 4

nodes_minimax = 0
t0 = time.time()
col_mm = get_best_move_minimax(demo_game, depth=DEPTH_COMPARE)
t_mm   = time.time() - t0
nm     = nodes_minimax

nodes_ab = 0
t0 = time.time()
col_ab = get_best_move_ab(demo_game, depth=DEPTH_COMPARE)
t_ab   = time.time() - t0
nab    = nodes_ab

print(f'Profundidad: d = {DEPTH_COMPARE}')
print(f'Minimax puro -> col: {col_mm} | nodos: {nm:>8,} | tiempo: {t_mm:.4f}s')
print(f'Alfa-Beta    -> col: {col_ab} | nodos: {nab:>8,} | tiempo: {t_ab:.4f}s')
print(f'Reduccion: {(1 - nab/nm)*100:.1f}% menos nodos con Alfa-Beta')

Tablero usado para la comparacion:
  0  1  2  3  4  5  6
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  Y  R  .  .  .
  R  Y  Y  Y  R  R  .

Profundidad: d = 4
Minimax puro -> col: 2 | nodos:    2,758 | tiempo: 0.6673s
Alfa-Beta    -> col: 2 | nodos:      475 | tiempo: 0.1029s
Reduccion: 82.8% menos nodos con Alfa-Beta


---
## Task 2.3 – Heurística y demos

### Función `evaluate_board(board)`

La heurística le asigna un puntaje al tablero desde la perspectiva de la IA (fichas Y). Se compone de dos partes:

**Control del centro:** la columna 3 aparece en más ventanas de 4 que cualquier otra columna, así que se le da +3 por cada ficha de la IA ahí.

**Ventanas de 4 celdas** — se escanean en las 4 direcciones:

| Contenido de la ventana | Puntaje |
|---|---|
| 4 fichas IA | +1000 |
| 3 IA + 1 vacío | +50 |
| 2 IA + 2 vacíos | +5 |
| 3 rival + 1 vacío | −80 |

El -80 es mayor en magnitud que el +50 para que la IA priorice bloquear antes que atacar cuando el rival está a un movimiento de ganar.

In [14]:
def get_random_move(game):
    valid = game.get_valid_columns()
    return random.choice(valid) if valid else None

In [15]:
# Demo 1: IA (Y, d=6) vs agente aleatorio (R)

AB_DEPTH = 6

def play_ai_vs_random(ab_depth=AB_DEPTH, verbose=True):
    game = Connect4()
    turn = 0  # 0=IA, 1=aleatorio

    if verbose:
        print(f'--- IA (Y, d={ab_depth}) vs Aleatorio (R) ---')
        game.print_board()

    while not game.is_terminal():
        if turn == 0:
            col = get_best_move_ab(game, depth=ab_depth)
            game.drop_piece(col, Connect4.AI)
            if verbose:
                print(f'IA (Y) -> col {col}')
                game.print_board()
            if game.check_win(Connect4.AI):
                if verbose: print('IA gana.')
                return 'AI'
        else:
            col = get_random_move(game)
            game.drop_piece(col, Connect4.PLAYER)
            if verbose:
                print(f'Aleatorio (R) -> col {col}')
                game.print_board()
            if game.check_win(Connect4.PLAYER):
                if verbose: print('Aleatorio gana.')
                return 'RANDOM'
        turn = 1 - turn

    if verbose: print('Empate.')
    return 'DRAW'

result = play_ai_vs_random()
print(f'Resultado: {result}')

--- IA (Y, d=6) vs Aleatorio (R) ---
  0  1  2  3  4  5  6
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .

IA (Y) -> col 3
  0  1  2  3  4  5  6
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  Y  .  .  .

Aleatorio (R) -> col 2
  0  1  2  3  4  5  6
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  R  Y  .  .  .

IA (Y) -> col 2
  0  1  2  3  4  5  6
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  Y  .  .  .  .
  .  .  R  Y  .  .  .

Aleatorio (R) -> col 3
  0  1  2  3  4  5  6
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  Y  R  .  .  .
  .  .  R  Y  .  .  .

IA (Y) -> col 2
  0  1  2  3  4  5  6
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .

In [ ]:
# Demo 2: Humano (R) vs IA (Y, d=6) — modo interactivo por consola

def play_human_vs_ai(ab_depth=AB_DEPTH):
    game = Connect4()
    print(f'--- Humano (R) vs IA (Y, d={ab_depth}) ---')
    print('Ingresa el numero de columna (0-6) en tu turno.')
    print('R = tu ficha | Y = IA | . = vacio')
    print()
    game.print_board()

    turn = 0  # 0=IA primero, 1=humano

    while not game.is_terminal():
        if turn == 0:
            print('IA pensando...')
            col = get_best_move_ab(game, depth=ab_depth)
            game.drop_piece(col, Connect4.AI)
            print(f'IA (Y) -> col {col}')
            game.print_board()
            if game.check_win(Connect4.AI):
                print('IA gana.')
                return
        else:
            valid = game.get_valid_columns()
            col = -1
            while col not in valid:
                try:
                    col = int(input(f'Tu turno (R) — columnas validas {valid}: '))
                    if col not in valid:
                        print('Columna invalida, intenta de nuevo.')
                except ValueError:
                    print('Ingresa un numero entero.')
            game.drop_piece(col, Connect4.PLAYER)
            print(f'Humano (R) -> col {col}')
            game.print_board()
            if game.check_win(Connect4.PLAYER):
                print('Humano gana.')
                return

        turn = 1 - turn

    print('Empate.')

play_human_vs_ai()

--- Humano (R) vs IA (Y, d=6) ---
Ingresa el numero de columna (0-6) en tu turno.
R = tu ficha | Y = IA | . = vacio

  0  1  2  3  4  5  6
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .

IA pensando...
IA (Y) -> col 3
  0  1  2  3  4  5  6
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  .  .  .  .
  .  .  .  Y  .  .  .

Ingresa un numero entero.
Ingresa un numero entero.
Ingresa un numero entero.
Ingresa un numero entero.
Ingresa un numero entero.
Ingresa un numero entero.
Ingresa un numero entero.
Ingresa un numero entero.
Ingresa un numero entero.
Ingresa un numero entero.
Ingresa un numero entero.
Ingresa un numero entero.
